In [1]:
!pip install emoji

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 9.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
from matplotlib import pyplot as plt
import numpy as np

import re
import string
import emoji

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

from sklearn.model_selection import RandomizedSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score, roc_curve, auc

In [3]:
train = pd.read_csv("train (2).csv")
test = pd.read_csv("test (1).csv")
val = pd.read_csv("dev.csv")

In [4]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [5]:
train = pd.concat([train, val], axis=0, ignore_index=True)

In [6]:
train.shape[0]

16000

In [7]:
train.duplicated().any()

np.False_

In [8]:
train.head()

,rewire_id,text,label_sexist,label_category,label_vector,split
0,sexism2022_english-16993,"Then, she's a keeper. 😉",not sexist,none,none,train
1,sexism2022_english-13149,This is like the Metallica video where the poo...,not sexist,none,none,train
2,sexism2022_english-13021,woman?,not sexist,none,none,train
3,sexism2022_english-14998,Unlicensed day care worker reportedly tells co...,not sexist,none,none,train
4,sexism2022_english-7228,[USER] Leg day is easy. Hot girls who wear min...,sexist,3. animosity,3.3 backhanded gendered compliments,train


In [9]:
train.drop(columns=['rewire_id', 'label_category', 'label_vector', 'split'], inplace=True)
test.drop(columns=['rewire_id', 'label_category', 'label_vector', 'split'], inplace=True)

In [10]:
def change_column_names(df, text_col, label_col):
    return df.rename(columns={ text_col: "text", label_col: "target"})

In [11]:
train = change_column_names(train, "text", "label_sexist")
test = change_column_names(test, "text", "label_sexist")

In [12]:
train.head()

,text,target
0,"Then, she's a keeper. 😉",not sexist
1,This is like the Metallica video where the poo...,not sexist
2,woman?,not sexist
3,Unlicensed day care worker reportedly tells co...,not sexist
4,[USER] Leg day is easy. Hot girls who wear min...,sexist


In [13]:
stop_words = set(stopwords.words("english"))
negations = {"no", "not", "nor", "never"}
stop_words = stop_words - negations

In [14]:
stemmer = PorterStemmer()

In [15]:
def clean_text(text: str) -> str:
    text = text.lower()

    text = emoji.demojize(text, delimiters=(" ", " "))

    text = re.sub(r"http\S+|www\S+", " URL ", text)
    text = re.sub(r"@\w+", " USER ", text)
    text = re.sub(r"\d+", " NUM ", text)

    text = re.sub(r"!{2,}", " EXCLAMATION ", text)
    text = re.sub(r"\?{2,}", " QUESTION ", text)

    punctuation = string.punctuation.replace("_", "")
    text = text.translate(str.maketrans("", "", punctuation))

    tokens = text.split()

    tokens = [
        stemmer.stem(token)
        for token in tokens
        if token not in stop_words
    ]

    return " ".join(tokens)


In [16]:
print(train['target'].value_counts())
print(test['target'].value_counts())

target
not sexist    12116
sexist         3884
Name: count, dtype: int64
target
not sexist    3030
sexist         970
Name: count, dtype: int64


In [17]:
train["clean_text"] = train["text"].apply(clean_text)
test["clean_text"] = test["text"].apply(clean_text)

In [18]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_df=0.9, sublinear_tf=True)

In [19]:
train.columns

Index(['text', 'target', 'clean_text'], dtype='object')

In [20]:
label_map = {
    "not sexist": 0,
    "sexist": 1
}

In [21]:
train["target"] = train["target"].str.strip().str.lower().map(label_map)
test["target"] = test["target"].str.strip().str.lower().map(label_map)

In [22]:
x_train = train['clean_text'].values
x_test = test['clean_text'].values

y_train = train['target'].values
y_test = test['target'].values

In [23]:
x_train_vec = vectorizer.fit_transform(x_train)
x_test_vec = vectorizer.transform(x_test)

In [24]:
log_reg = LogisticRegression(
    solver='saga',
    max_iter=3000,
    n_jobs=-1
)

In [25]:
svc = LinearSVC(max_iter=3000)

In [26]:
rfc = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

In [27]:
logReg_param_dist = {
    'C': np.logspace(-3, 2, 20),
    'penalty': ['l1', 'l2']
}
rfc_param_dist = {
    'n_estimators': [200, 300, 500],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5, 10]
}
svc_param_dist = {
    'C': np.logspace(-3, 2, 20)
}

In [28]:
rand_lr = RandomizedSearchCV(
    log_reg,
    logReg_param_dist,
    n_iter=20,
    cv=5,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)
rand_rfc = RandomizedSearchCV(
    rfc,
    rfc_param_dist,
    n_iter=30,
    cv=5,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)
rand_svc = RandomizedSearchCV(
    svc,
    svc_param_dist,
    n_iter=20,
    cv=5,
    scoring='f1',
    random_state=42,
    n_jobs=-1
)

In [30]:
rand_lr.fit(x_train_vec, y_train)

RandomizedSearchCV(cv=5,
                   estimator=LogisticRegression(max_iter=3000, n_jobs=-1,
                                                solver='saga'),
                   n_iter=20, n_jobs=-1,
                   param_distributions={'C': array([1.00000000e-03, 1.83298071e-03, 3.35981829e-03, 6.15848211e-03,
       1.12883789e-02, 2.06913808e-02, 3.79269019e-02, 6.95192796e-02,
       1.27427499e-01, 2.33572147e-01, 4.28133240e-01, 7.84759970e-01,
       1.43844989e+00, 2.63665090e+00, 4.83293024e+00, 8.85866790e+00,
       1.62377674e+01, 2.97635144e+01, 5.45559478e+01, 1.00000000e+02]),
                                        'penalty': ['l1', 'l2']},
                   random_state=42, scoring='f1')

In [31]:
rand_rfc.fit(x_train_vec, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_search.py:317: UserWarning: The total space of parameters 27 is smaller than n_iter=30. Running 27 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
rand_lr.fit(x_train_vec, y_train)
rand_svc.fit(x_train_vec, y_train)
rand_rfc.fit(x_train_vec, y_train)

In [32]:
print(rand_lr.best_params_)
print(rand_svc.best_params_)
print(rand_rfc.best_params_)

{'penalty': 'l1', 'C': np.float64(2.636650898730358)}

In [40]:
lr_model = rand_lr.best_estimator_
svc_model = rand_lr.best_estimator_
rfc_model = rand_lr.best_estimator_

In [41]:
y_lr_pred = lr_model.predict(x_test_vec)
y_svc_pred = svc_model.predict(x_test_vec)
y_rfc_pred = lr_mrfc_modelodel.predict(x_test_vec)

In [42]:
print(classification_report(y_true=y_test, y_pred=y_lr_pred))
print(classification_report(y_true=y_test, y_pred=y_svc_pred))
print(classification_report(y_true=y_test, y_pred=y_rfc_pred))

              precision    recall  f1-score   support

           0       0.87      0.94      0.90      3030
           1       0.74      0.55      0.63       970

    accuracy                           0.85      4000
   macro avg       0.81      0.75      0.77      4000
weighted avg       0.84      0.85      0.84      4000



In [ ]:
# metrics = ["F1", "Recall", "Precision", "Accuracy"]
# f1_scores = [round(f1_score(y_true=y_test, y_pred=logReg_pred), 2), round(f1_score(y_true=y_test, y_pred=svc_pred), 2), round(f1_score(y_true=y_test, y_pred=rfc_pred), 2)]
# recall_scores = [round(recall_score(y_true=y_test, y_pred=logReg_pred), 2), round(recall_score(y_true=y_test, y_pred=svc_pred), 2), round(recall_score(y_true=y_test, y_pred=rfc_pred), 2)]
# precision_scores = [round(precision_score(y_true=y_test, y_pred=logReg_pred), 2), round(precision_score(y_true=y_test, y_pred=svc_pred), 2), round(precision_score(y_true=y_test, y_pred=rfc_pred), 2)]
# accuracy_scores = [round(accuracy_score(y_true=y_test, y_pred=logReg_pred), 2), round(accuracy_score(y_true=y_test, y_pred=svc_pred), 2), round(accuracy_score(y_true=y_test, y_pred=rfc_pred), 2)]

# scores = {
#     "Logistic Regression": [f1_scores[0], recall_scores[0], precision_scores[0], accuracy_scores[0]],
#     "SVC": [f1_scores[1], recall_scores[1], precision_scores[1], accuracy_scores[1]],
#     "RandomForest": [f1_scores[2], recall_scores[2], precision_scores[2], accuracy_scores[2]]
# }

In [ ]:
# fig, axs = plt.subplots(1, 3, figsize=(15, 5), sharey=True)

# for ax, (model, vals) in zip(axs, scores.items()):
#     ax.bar(metrics, vals)
#     ax.set_title(model)
#     ax.set_ylim(0, 1)

#     for i, v in enumerate(vals):
#         ax.text(i, v + 0.01, round(v, 2), ha="center")

# fig.suptitle("Model-wise Metric Comparison", fontsize=14)
# plt.tight_layout()
# plt.show()


In [ ]:
# logreg_scores = logReg.predict_proba(x_test_vec)[:, 1]
# rfc_scores = rfc.predict_proba(x_test_vec)[:, 1]
# svc_scores = svc.predict_proba(x_test_vec)[:, 1]

In [ ]:
# fpr_lr, tpr_lr, _ = roc_curve(y_test, logreg_scores)
# fpr_svc, tpr_svc, _ = roc_curve(y_test, svc_scores)
# fpr_rfc, tpr_rfc, _ = roc_curve(y_test, rfc_scores)

# auc_lr = auc(fpr_lr, tpr_lr)
# auc_svc = auc(fpr_svc, tpr_svc)
# auc_rfc = auc(fpr_rfc, tpr_rfc)

In [ ]:

# plt.figure(figsize=(8,6))

# plt.plot(fpr_lr, tpr_lr, label=f"LogReg (AUC={auc_lr:.2f})")
# plt.plot(fpr_svc, tpr_svc, label=f"SVC (AUC={auc_svc:.2f})")
# plt.plot(fpr_rfc, tpr_rfc, label=f"RandomForest (AUC={auc_rfc:.2f})")

# plt.plot([0, 1], [0, 1], linestyle="--", label="Random Guess")

# plt.xlabel("False Positive Rate")
# plt.ylabel("True Positive Rate")
# plt.title("ROC Curve Comparison")
# plt.legend()
# plt.show()